# Ablation Study

Without External Features

In [ ]:
import pandas as pd

df = pd.read_csv("../data/cleaned_ai4i.csv")

print(df.shape)

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.feature_engineering import generate_rolling_features

In [ ]:
df_roll = generate_rolling_features(df)

print(df_roll.shape)

In [ ]:
rolling_cols = [col for col in df_roll.columns if "rolling" in col]

print(len(rolling_cols))
print(rolling_cols)

In [ ]:
base_features = [
    col for col in df_roll.columns
    if col not in [
        'Machine failure',
        'UDI',
        'Product ID',
        'Type',
        'TWF',
        'HDF',
        'PWF',
        'OSF',
        'RNF'
    ]
]

print(len(base_features))


In [ ]:
X = df_roll[base_features]

y = df_roll['Machine failure']

print(X.shape)
print(y.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
from sklearn.model_selection import cross_validate

scores = cross_validate(
    rf,
    X,
    y,
    cv=cv,
    scoring='f1_macro'
)

macro_f1 = scores['test_score'].mean()

print("Macro F1 Score =", round(macro_f1,4))

## Without External Features

Model Used:
RandomForestClassifier

Validation Method:
5-Fold StratifiedKFold Cross Validation

Feature Set:
- Type_enc
- Original IoT Sensor Features
- Rolling Mean Features
- Rolling Standard Deviation Features
- Rolling Variance Features

Result:
Macro F1 Score = 0.8104

Conclusion:
The baseline Random Forest model achieved a Macro F1 Score of 0.8104 using only internal IoT telemetry and rolling statistical features. This result will be used as a benchmark before adding external contextual features.

## With External Features

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
from src.feature_sets import ext_features

print(len(ext_features))
print(ext_features)

In [ ]:
df_roll = generate_rolling_features(df)

print(df_roll.shape)

In [ ]:
print(df_roll.head())

In [ ]:
print(df_roll.columns.tolist())

In [ ]:
print(df.columns.tolist())

In [ ]:
print(df.shape)

In [ ]:
external_cols = [
    "ambient_temp_C",
    "factory_load_pct",
    "humidity_pct"
]

for col in external_cols:
    print(col, "->", col in df.columns)

In [ ]:
import numpy as np

np.random.seed(42)

df_roll["ambient_temp_C"] = np.random.normal(
    loc=28,
    scale=5,
    size=len(df_roll)
)

df_roll["factory_load_pct"] = np.random.uniform(
    50,
    100,
    size=len(df_roll)
)

df_roll["humidity_pct"] = np.random.normal(
    loc=60,
    scale=10,
    size=len(df_roll)
)

In [ ]:
print(
    df_roll[
        [
            "ambient_temp_C",
            "factory_load_pct",
            "humidity_pct"
        ]
    ].head()
)

In [ ]:
from src.feature_sets import ext_features

X_ext = df_roll[ext_features]

y_ext = df_roll["Machine failure"]

print(X_ext.shape)
print(y_ext.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

print(rf)

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(cv)

In [ ]:
from sklearn.model_selection import cross_validate

scores_ext = cross_validate(
    rf,
    X_ext,
    y_ext,
    cv=cv,
    scoring="f1_macro"
)

ext_macro_f1 = scores_ext["test_score"].mean()

print("Fold Scores:", scores_ext["test_score"])
print("With External Features Macro F1 =", round(ext_macro_f1, 4))

In [ ]:
print(ext_macro_f1)

In [ ]:
print("Without External Features =", round(macro_f1, 4))
print("With External Features    =", round(ext_macro_f1, 4))

## Conclusion

Without External Features:
Macro F1 = 0.8008

With External Features:
Macro F1 = 0.5027

The addition of simulated external context features did not improve model performance in this experiment. The Random Forest model achieved a lower Macro F1 score when external features were included.